# Task 4: Visual Search

In [ ]:


from pathlib import Path

import numpy as np
import pandas as pd
import torch
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from PIL import Image, ImageOps
from torchvision import transforms


In [6]:
# Works whether Jupyter starts in the repository root or in notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'preprocessed_datasets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'preprocessed_datasets' / 'train' / 'styles_train.csv'
IMAGE_DIR = PROJECT_ROOT / 'preprocessed_datasets' / 'train' / 'images_train'
SPLIT_DIR = PROJECT_ROOT / 'splits' / 'task1'

df = pd.read_csv(DATA_PATH)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 37745 entries, 0 to 37744
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   id                  37745 non-null  int64
 1   gender              37745 non-null  str  
 2   masterCategory      37745 non-null  str  
 3   subCategory         37745 non-null  str  
 4   articleType         37745 non-null  str  
 5   baseColour          37745 non-null  str  
 6   season              37745 non-null  str  
 7   year                37745 non-null  int64
 8   usage               37745 non-null  str  
 9   productDisplayName  37745 non-null  str  
dtypes: int64(2), str(8)
memory usage: 2.9 MB


In [7]:
stratify_cols = [
    "masterCategory", "subCategory", "baseColour", "year",
    "articleType", "season", "gender", "usage",
]

# One binary feature per category across all chosen columns
stratify_features = pd.get_dummies(
    df[stratify_cols].fillna("missing").astype(str),
    prefix=stratify_cols,
)

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42,
)

train_idx, test_idx = next(splitter.split(df, stratify_features))
train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print('Train:', train_df.shape)
print('Test: ', test_df.shape)

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(SPLIT_DIR / 'train.csv', index=False)
test_df.to_csv(SPLIT_DIR / 'test.csv', index=False)

Train: (33970, 10)
Test:  (3775, 10)


## Image preprocessing

The images in `preprocessed_datasets` are assumed to have already been standardised: orientation corrected and converted to RGB. They are letterbox-resized to the shared ResNet-18 input size of 128x128: the aspect ratio is preserved and unused space is padded white.

RGB normalisation is fitted only on the training split. Augmentation is used only for training; validation, test, gallery, and query images always use the deterministic evaluation transform.

In [8]:
RESNET_INPUT_SIZE = (128, 128)


class LetterboxResize:
    """Resize to fit within a canvas, padding instead of stretching or cropping."""

    def __init__(self, size, fill=(255, 255, 255)):
        self.size = tuple(size)
        self.fill = fill

    def __call__(self, image):
        return ImageOps.pad(
            image.convert('RGB'),
            self.size,
            method=Image.Resampling.BILINEAR,
            color=self.fill,
            centering=(0.5, 0.5),
        )

letterbox_to_resnet = LetterboxResize(RESNET_INPUT_SIZE)

def compute_rgb_mean_std(record_ids):
    """Calculate per-channel RGB statistics using training images only."""
    channel_sum = torch.zeros(3, dtype=torch.float64)
    channel_sum_sq = torch.zeros(3, dtype=torch.float64)
    pixel_count = 0

    for record_id in record_ids:
        path = IMAGE_DIR / f'{int(record_id)}.jpg'

        if not path.exists():
            raise FileNotFoundError(f'Missing image: {path}')
        
        with Image.open(path) as image:
            tensor = transforms.ToTensor()(letterbox_to_resnet(image)).to(torch.float64)

        channel_sum += tensor.sum(dim=(1, 2))
        channel_sum_sq += (tensor ** 2).sum(dim=(1, 2))
        pixel_count += tensor.shape[1] * tensor.shape[2]

    mean = channel_sum / pixel_count
    std = torch.sqrt(channel_sum_sq / pixel_count - mean ** 2)
    return mean.float().tolist(), std.float().tolist()

train_mean, train_std = compute_rgb_mean_std(train_df['id'])
print('Training RGB mean:', np.round(train_mean, 4))
print('Training RGB std: ', np.round(train_std, 4))

Training RGB mean: [0.8862 0.8742 0.8695]
Training RGB std:  [0.2397 0.2513 0.2546]


In [9]:
# Triplet Margin, SupCon, Multi-Similarity, and ArcFace training.
# Mild geometric augmentation preserves the full product and its colour cues.
metric_train_transform = transforms.Compose([
    letterbox_to_resnet,
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(
        degrees=5,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
        fill=(255, 255, 255),
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=train_mean, std=train_std),
])

# Use unchanged for validation, test, gallery, and query images.
metric_eval_transform = transforms.Compose([
    letterbox_to_resnet,
    transforms.ToTensor(),
    transforms.Normalize(mean=train_mean, std=train_std),
])

# CAE input and MSE reconstruction target are letterboxed to 128x128 in [0, 1].
# Pair this with a decoder ending in Sigmoid().
cae_transform = transforms.Compose([
    letterbox_to_resnet,
    transforms.ToTensor(),
])

### Transform assignment

- **CAE:** use `cae_transform` for both input and reconstruction target; start without augmentation.
- **Triplet Margin, Multi-Similarity, ArcFace:** use `metric_train_transform` during training and `metric_eval_transform` otherwise.
- **SupCon:** apply `metric_train_transform` twice independently to each training image for its two views; use `metric_eval_transform` for evaluation.

Random resized crops, strong rotations, and strong colour jitter are intentionally excluded because they can remove product details or distort colour information relevant to visual search.